# Пример 4. Базовый агент: роль, цикл исполнения и трасса

**Модуль I · Тема 3** — базовый агент: исполняемая среда, цикл работы и ролевая системная инструкция

Здесь собирается первый настоящий агент — **вручную, без каркаса**. Это принципиально: пока цикл не написан своими руками, он остаётся «чёрным ящиком». Каркас (LangGraph) появится в примере 5, и сравнение будет осмысленным.

Рабочее определение курса: **агент = исполняемая среда (runtime), объединяющая языковую модель, системную инструкцию, инструменты, память, механизмы защиты и планирования.** Ниже видно, что «runtime» — это буквально цикл `while`.

### Что показано
1. Ролевая системная инструкция из четырёх обязательных блоков.
2. Уровни сообщений: `system` / `user` / `assistant` / `tool`.
3. Цикл исполнения с ограничением итераций и обработкой ошибок.
4. Полная трасса шагов.
5. Три сценария: успех, отказ, деградация.

### Доступ к модели
Параметры — из переменных окружения `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
import os, json, time
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("LLM_API_KEY"),
                base_url=os.environ.get("LLM_BASE_URL"))
MODEL = os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini")
print("Модель:", MODEL)

Модель: openai/gpt-4.1-mini


## 1. Предметная область и инструменты

Учебный домен — поддержка интернет-магазина. Инструменты намеренно простые: здесь они нужны как повод для **ветвления** в цикле. Проектирование контрактов инструментов — задача модуля II.

In [2]:
ORDERS = {
    "ORD-1002": {"status": "отправлен", "items": "USB-C хаб ×2", "total": 4980},
    "ORD-1003": {"status": "в сборке",  "items": "наушники ×1",  "total": 4990},
}
POLICIES = {
    "возврат":  "Возврат в течение 14 дней с момента получения при сохранении упаковки.",
    "доставка": "Стандартная доставка 3–5 рабочих дней.",
}

def get_order(order_id: str) -> dict:
    """Информация о заказе по идентификатору."""
    if order_id not in ORDERS:
        return {"error": "not_found", "order_id": order_id}
    return {"order_id": order_id, **ORDERS[order_id]}

def get_policy(topic: str) -> dict:
    """Правила магазина: возврат или доставка."""
    if topic not in POLICIES:
        return {"error": "unknown_topic", "available": list(POLICIES)}
    return {"topic": topic, "text": POLICIES[topic]}

TOOL_IMPL = {"get_order": get_order, "get_policy": get_policy}

# Описание инструментов для модели (схема в формате OpenAI tools)
TOOLS_SPEC = [
    {"type": "function", "function": {
        "name": "get_order", "description": "Получить информацию о заказе по его идентификатору.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string", "description": "например ORD-1002"}},
                       "required": ["order_id"]}}},
    {"type": "function", "function": {
        "name": "get_policy", "description": "Получить правила магазина по теме.",
        "parameters": {"type": "object",
                       "properties": {"topic": {"type": "string", "enum": ["возврат", "доставка"]}},
                       "required": ["topic"]}}},
]
print("Инструментов:", len(TOOLS_SPEC))

Инструментов: 2


## 2. Ролевая системная инструкция

Системная инструкция — **носитель роли**. Она должна содержать четыре блока; отсутствие любого из них приводит к предсказуемым дефектам поведения.

| Блок | Назначение | Что сломается без него |
|------|-----------|------------------------|
| **Цель** | что агент обязан обеспечить | агент берётся за посторонние задачи |
| **Ограничения и запреты** | чего не делает никогда | выдаёт чужие данные, обещает несуществующее |
| **Стилистика** | как формулирует ответ | ответы непригодны для интерфейса |
| **Политика решений** | как вести себя в неоднозначной ситуации | догадывается вместо уточнения; выдумывает вместо отказа |

Обратите внимание на политику решений: именно она превращает модель в **предсказуемого исполнителя роли**.

In [3]:
SYSTEM_PROMPT = """Ты — агент поддержки интернет-магазина «ШопБот».

ЦЕЛЬ.
Отвечать на вопросы клиентов о статусе заказов и правилах магазина.

ОГРАНИЧЕНИЯ И ЗАПРЕТЫ.
- Не сообщай никаких данных, которых нет в результатах инструментов.
- Не обещай сроки, скидки и компенсации.
- Не отвечай на вопросы вне тематики магазина (погода, политика, программирование).

СТИЛИСТИКА.
- Русский язык, вежливо, не более 3 предложений.
- Всегда указывай идентификатор заказа, если речь о нём.

ПОЛИТИКА ПРИНЯТИЯ РЕШЕНИЙ.
- Нужны данные о заказе или правилах — вызови соответствующий инструмент, не отвечай по памяти.
- Инструмент вернул ошибку not_found — сообщи, что заказ не найден, и попроси проверить номер.
- Данных не хватает — задай один уточняющий вопрос, не догадывайся.
- Вопрос вне тематики — вежливо откажи и напомни о своей роли."""
print(SYSTEM_PROMPT)

Ты — агент поддержки интернет-магазина «ШопБот».

ЦЕЛЬ.
Отвечать на вопросы клиентов о статусе заказов и правилах магазина.

ОГРАНИЧЕНИЯ И ЗАПРЕТЫ.
- Не сообщай никаких данных, которых нет в результатах инструментов.
- Не обещай сроки, скидки и компенсации.
- Не отвечай на вопросы вне тематики магазина (погода, политика, программирование).

СТИЛИСТИКА.
- Русский язык, вежливо, не более 3 предложений.
- Всегда указывай идентификатор заказа, если речь о нём.

ПОЛИТИКА ПРИНЯТИЯ РЕШЕНИЙ.
- Нужны данные о заказе или правилах — вызови соответствующий инструмент, не отвечай по памяти.
- Инструмент вернул ошибку not_found — сообщи, что заказ не найден, и попроси проверить номер.
- Данных не хватает — задай один уточняющий вопрос, не догадывайся.
- Вопрос вне тематики — вежливо откажи и напомни о своей роли.


## 3. Уровни сообщений диалога

Диалог с моделью — это список сообщений, у каждого своя **роль**:

| Роль | Кто формирует | Назначение |
|------|---------------|-----------|
| `system` | разработчик | роль агента; задаётся один раз, имеет наивысший приоритет |
| `user` | пользователь | запрос; **недоверенные данные** |
| `assistant` | модель | ответ или запрос на вызов инструмента (`tool_calls`) |
| `tool` | ваш код | результат исполнения инструмента, возвращаемый модели |

Ключевой момент безопасности: `system` и `user` — **разные уровни доверия**. Указания разработчика идут в `system`; пользовательский ввод никогда не должен туда попадать (иначе пользователь сможет переопределить роль — модуль II, тема 7).

## 4. Цикл исполнения

Вот то, что отличает агента от одиночного запроса. Одиночный запрос — «спросили → получили текст». Агент — **цикл**:

1. отправить историю сообщений модели;
2. посмотреть на ответ: это финальный текст или запрос на вызов инструмента?
3. если инструмент — исполнить его, добавить результат как сообщение `tool`, вернуться к шагу 1;
4. если текст — завершить.

Два обязательных предохранителя: **лимит итераций** (иначе агент может зациклиться) и **обработка ошибок инструмента** (иначе падение программы).

Попутно собираем **трассу** — структурированный протокол шагов.

In [4]:
def run_agent(user_query: str, max_iterations: int = 5, verbose: bool = True):
    """Базовый агент: цикл «модель <-> инструменты» с трассой шагов."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_query}]
    trace, stop_reason = [], "completed"

    for step in range(1, max_iterations + 1):
        t0 = time.perf_counter()
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS_SPEC, temperature=0)
        dt = time.perf_counter() - t0
        msg = resp.choices[0].message

        trace.append({"step": step, "type": "llm_call",
                      "tool_calls": [tc.function.name for tc in (msg.tool_calls or [])],
                      "content": (msg.content or "")[:200],
                      "tokens_in": resp.usage.prompt_tokens,
                      "tokens_out": resp.usage.completion_tokens,
                      "seconds": round(dt, 2)})

        # Модель не запросила инструментов -> это финальный ответ, цикл завершается
        if not msg.tool_calls:
            if verbose: print(f"[шаг {step}] модель: финальный ответ")
            return {"answer": msg.content, "trace": trace, "stop_reason": stop_reason}

        messages.append(msg)   # ответ модели с запросом вызовов

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments)
                result = TOOL_IMPL[name](**args)          # исполнение инструмента
                error = None
            except Exception as e:                        # предохранитель: агент не падает
                result, error = {"error": "tool_failed", "detail": str(e)}, type(e).__name__

            if verbose: print(f"[шаг {step}] инструмент {name}({args}) -> {result}")
            trace.append({"step": step, "type": "tool_call", "tool": name,
                          "args": args, "result": result, "error": error})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})

    # Лимит исчерпан — обязательный выход, иначе цикл бесконечен
    return {"answer": "Не удалось завершить обработку запроса за отведённое число шагов.",
            "trace": trace, "stop_reason": "max_iterations_exceeded"}

print("Агент готов.")

Агент готов.


## 5. Сценарий 1 — успех

Запрос требует данных из двух разных инструментов. Проследите по выводу: модель сама решает, какие инструменты вызвать, и цикл делает **несколько** обращений к ней.

In [5]:
r1 = run_agent("Что с заказом ORD-1002 и как оформить возврат?")
print("\nОТВЕТ:", r1["answer"])
print("причина остановки:", r1["stop_reason"], "| шагов в трассе:", len(r1["trace"]))

[шаг 1] инструмент get_order({'order_id': 'ORD-1002'}) -> {'order_id': 'ORD-1002', 'status': 'отправлен', 'items': 'USB-C хаб ×2', 'total': 4980}
[шаг 1] инструмент get_policy({'topic': 'возврат'}) -> {'topic': 'возврат', 'text': 'Возврат в течение 14 дней с момента получения при сохранении упаковки.'}


[шаг 2] модель: финальный ответ

ОТВЕТ: Заказ ORD-1002 со статусом "отправлен", в нем USB-C хаб ×2 на сумму 4980 рублей. Возврат возможен в течение 14 дней с момента получения при сохранении упаковки. Если нужна помощь с оформлением возврата, сообщите, пожалуйста.
причина остановки: completed | шагов в трассе: 4


Одним запросом к модели это было бы невозможно: чтобы ответить, нужно **сначала** получить данные, а какие именно — становится известно только после первого обращения к модели. Это и есть практическое отличие цикла от одиночного запроса.

## 6. Сценарий 2 — отказ

Запрос вне роли. Корректный отказ — **правильный результат**, а не сбой. Обратите внимание: инструменты не вызываются, цикл завершается за один шаг.

In [6]:
r2 = run_agent("Напиши функцию быстрой сортировки на Python.")
print("\nОТВЕТ:", r2["answer"])
print("вызовов инструментов:", sum(1 for s in r2["trace"] if s["type"] == "tool_call"))

[шаг 1] модель: финальный ответ

ОТВЕТ: Извините, я могу помочь только с вопросами, связанными с заказами и правилами магазина. Если у вас есть вопросы по этим темам, пожалуйста, задайте их.
вызовов инструментов: 0


## 7. Сценарий 3 — деградация

Инструмент возвращает ошибку (`not_found`). Агент обязан корректно сообщить о проблеме, а не выдумать статус — это прямая проверка блока «политика принятия решений» из системной инструкции.

In [7]:
r3 = run_agent("Какой статус у заказа ORD-99999?")
print("\nОТВЕТ:", r3["answer"])

[шаг 1] инструмент get_order({'order_id': 'ORD-99999'}) -> {'error': 'not_found', 'order_id': 'ORD-99999'}


[шаг 2] модель: финальный ответ

ОТВЕТ: Заказ с идентификатором ORD-99999 не найден в нашей системе. Пожалуйста, проверьте номер заказа и сообщите его снова.


## 8. Трасса исполнения

Трасса — структурированный протокол работы агента. Её назначение: по трассе постороннему должно быть понятно, **почему** агент выдал именно такой ответ. Это основа отладки и оценки качества (модуль IV).

In [8]:
import pandas as pd

def show_trace(result):
    rows = []
    for s in result["trace"]:
        if s["type"] == "llm_call":
            rows.append({"шаг": s["step"], "тип": "обращение к модели",
                         "детали": ("запросила: " + ", ".join(s["tool_calls"])) if s["tool_calls"]
                                   else "финальный ответ",
                         "ток. вх": s["tokens_in"], "ток. вых": s["tokens_out"], "сек": s["seconds"]})
        else:
            rows.append({"шаг": s["step"], "тип": f"инструмент {s['tool']}",
                         "детали": str(s["result"])[:60], "ток. вх": "", "ток. вых": "", "сек": ""})
    return pd.DataFrame(rows)

print("=== ТРАССА: сценарий 1 (успех) ===")
print(show_trace(r1).to_string(index=False))
print("\n=== ТРАССА: сценарий 3 (деградация) ===")
print(show_trace(r3).to_string(index=False))

=== ТРАССА: сценарий 1 (успех) ===
 шаг                   тип                                                       детали ток. вх ток. вых   сек
   1    обращение к модели                             запросила: get_order, get_policy     343       51  2.83
   1  инструмент get_order {'order_id': 'ORD-1002', 'status': 'отправлен', 'items': 'US                       
   1 инструмент get_policy {'topic': 'возврат', 'text': 'Возврат в течение 14 дней с мо                       
   2    обращение к модели                                              финальный ответ     455       64  2.44

=== ТРАССА: сценарий 3 (деградация) ===
 шаг                  тип                                          детали ток. вх ток. вых   сек
   1   обращение к модели                            запросила: get_order     339       19  2.12
   1 инструмент get_order {'error': 'not_found', 'order_id': 'ORD-99999'}                       
   2   обращение к модели                                 финальный ответ     

Трасса сохраняется в файл — это артефакт, который сдаётся в лабораторной работе.

In [9]:
with open("trace_example.json", "w", encoding="utf-8") as f:
    json.dump({"query": "Что с заказом ORD-1002 и как оформить возврат?",
               "answer": r1["answer"], "stop_reason": r1["stop_reason"],
               "trace": r1["trace"]}, f, ensure_ascii=False, indent=2)

total_in  = sum(s.get("tokens_in", 0)  for s in r1["trace"] if s["type"] == "llm_call")
total_out = sum(s.get("tokens_out", 0) for s in r1["trace"] if s["type"] == "llm_call")
print("Трасса сохранена: trace_example.json")
print(f"Суммарно за сценарий 1: {total_in} вх. + {total_out} вых. токенов, "
      f"обращений к модели: {sum(1 for s in r1['trace'] if s['type']=='llm_call')}")

Трасса сохранена: trace_example.json
Суммарно за сценарий 1: 798 вх. + 115 вых. токенов, обращений к модели: 2


**Обратите внимание на стоимость.** Один запрос пользователя обошёлся в несколько обращений к модели, и история сообщений растёт с каждым шагом — значит, входные токены на каждом шаге **накапливаются**. Это ключевой фактор экономики агентов (модули III и IV).

## 9. Зачем нужен лимит итераций

Проверим предохранитель: искусственно ограничим цикл одним шагом на задаче, которая требует нескольких. Агент обязан завершиться штатно, а не зациклиться.

In [10]:
r4 = run_agent("Что с заказом ORD-1002 и как оформить возврат?", max_iterations=1, verbose=False)
print("причина остановки:", r4["stop_reason"])
print("ответ:", r4["answer"])

причина остановки: max_iterations_exceeded
ответ: Не удалось завершить обработку запроса за отведённое число шагов.


Без этого предохранителя агент, попавший в петлю «вызываю инструмент → не устраивает результат → вызываю снова», работал бы до исчерпания квоты. Лимит итераций — обязательный элемент любого агентного цикла.

## Итоги

- **Агент = runtime**, и на этом уровне runtime — это цикл `while` вокруг вызова модели.
- Отличие от одиночного запроса: агент **сам решает**, какие данные ему нужны, и получает их за несколько шагов.
- **Системная инструкция** несёт роль; блок «политика принятия решений» определяет поведение в неоднозначных и отказных ситуациях.
- **Уровни сообщений** разделяют доверие: `system` — разработчик, `user` — недоверенный ввод, `tool` — результаты инструментов.
- Обязательные предохранители: **лимит итераций** и **обработка ошибок инструмента**.
- **Трасса** делает поведение агента объяснимым и является артефактом отладки и оценки.
- Корректный **отказ** — такой же валидный результат, как и ответ.

Дальше: тот же агент, собранный на каркасе LangGraph — пример 5. **В лабораторной работе ([КИМ-1.3](../../kim-lab-03.md))** этот цикл реализуется для вашей роли.